In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU not available")

True
Tesla T4


In [ ]:
!pip install ultralytics onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.1 MB/s eta 0:00:00


In [ ]:
import torch
import ultralytics
import onnx

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("GPU available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.183
PyTorch: 2.11.0+cu130
ONNX: 1.23.2
GPU available: True


In [ ]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Train on VisDrone using GPU
model.train(
    data="VisDrone.yaml",
    epochs=30,
    imgsz=640,
    device=0
)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overla

In [ ]:


# Load the best trained model
model = YOLO("runs/detect/train/weights/best.pt")

# Validate the model
metrics = model.val(
    data="VisDrone.yaml",
    split="val",
    imgsz=640,
    device=0
)

# Report overall metrics
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

# Report per-class AP50
for i, name in model.names.items():
    print(f"{name}: AP50={metrics.box.ap50[i]:.4f}")

# Export the trained model to ONNX
model.export(format="onnx")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,599,238 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3336.5±832.8 MB/s, size: 153.1 KB)
val: Scanning /content/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 209.0Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=317 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.1it/s 11.3s
                   all        548      38759      0.424      0.312      0.289      0.161
            pedestrian        520       8844      0.406      0.354      0.308     

'runs/detect/train/weights/best.onnx'

In [ ]:
from ultralytics import YOLO

# Load the exported ONNX model
model = YOLO("runs/detect/train/weights/best.onnx", task="detect")

# Validate the ONNX model on the VisDrone validation split
metrics = model.val(
    data="VisDrone.yaml",
    split="val",
    imgsz=640,
    device=0
)

# Report overall metrics
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

# Report per-class AP50
for i, name in model.names.items():
    print(f"{name}: AP50={metrics.box.ap50[i]:.4f}")




Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Loading runs/detect/train/weights/best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 5 packages in 1.78s
Prepared 1 package in 12.90s
Installed 1 package in 53ms
 + onnxruntime-gpu==1.31.0

requirements: AutoUpdate success ✅ 14.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Using ONNX Runtime 1.31.0 with CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2306.8±694.8 MB/s, size: 137.6 KB)
val: Scanning /content/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 76.6Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Destination folder
drive_dir = Path("/content/drive/MyDrive/VisDrone_Results")
drive_dir.mkdir(parents=True, exist_ok=True)

# Folders to save
folders = [
    Path("/content/datasets"),
    Path("/content/runs"),
    Path("/content/weights")
]

# Copy folders to Drive
for folder in folders:
    if folder.exists():
        destination = drive_dir / folder.name

        shutil.copytree(
            folder,
            destination,
            dirs_exist_ok=True
        )

        print(f"Saved: {folder} -> {destination}")
    else:
        print(f"Folder not found, skipped: {folder}")

print("Copy completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/datasets -> /content/drive/MyDrive/VisDrone_Results/datasets
Saved: /content/runs -> /content/drive/MyDrive/VisDrone_Results/runs
Saved: /content/weights -> /content/drive/MyDrive/VisDrone_Results/weights
Copy completed.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

destination = Path("/content/drive/MyDrive/VisDrone_Results")
destination.mkdir(parents=True, exist_ok=True)

for name in ["datasets", "runs", "weights"]:
    source = Path("/content") / name
    target = destination / name

    print(f"\nChecking {source}")

    if source.exists():
        shutil.copytree(source, target, dirs_exist_ok=True)
        print(f"✅ Copied {name}")
        print(f"📁 Drive path: {target}")
    else:
        print(f"⚠️ Source folder not found: {source}")

print("\nDrive folder contents:")
for item in destination.iterdir():
    print(item.name, "— folder" if item.is_dir() else "— file")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Checking /content/datasets
✅ Copied datasets
📁 Drive path: /content/drive/MyDrive/VisDrone_Results/datasets

Checking /content/runs
✅ Copied runs
📁 Drive path: /content/drive/MyDrive/VisDrone_Results/runs

Checking /content/weights
✅ Copied weights
📁 Drive path: /content/drive/MyDrive/VisDrone_Results/weights

Drive folder contents:
datasets — folder
runs — folder
weights — folder
